#Homework 4
Sarayu Rao

In [0]:
#Pyspark Imports
from pyspark.sql.functions import col, round, avg, upper, substring, when, length, floor, datediff, current_date, max, min, sum, when, regexp_extract, lit
import pyspark.sql.functions as F
import pandas as pd
from pyspark.sql import Window

#ML imports
from sklearn.model_selection import train_test_split 
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import os
import matplotlib.pyplot as plt
import mlflow.sklearn
import seaborn as sns
from sklearn.metrics import f1_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay, precision_score, recall_score, accuracy_score
import tempfile
from sklearn.preprocessing import StandardScaler

##Main Question
My predictions will be whether a driver won the race. 
For this assignment, I will be utilizing the same dataset setup from homework 3, but adding a column indicating whether a driver won the race. 

##Preparing the Data

In [0]:
#Load pitstop dataset
df_pitstops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)

#Cast necessary columns to integers
df_pitstops = df_pitstops.withColumns({"raceId": col("raceId").cast("int"),
                                      "driverId": col("driverId").cast("int"),
                                      "milliseconds": col("milliseconds").cast("int")})

#Get each driver's average pitstop time for each race
df_avg_pit = df_pitstops.groupBy("raceId","driverId").avg("milliseconds")

#Load results dataset
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)

#Join average pitstop time for each driver's race and rename column
df_race_results = df_results.join(df_avg_pit, on = ["raceId","driverId"])
df_race_results = df_race_results.withColumnRenamed("avg(milliseconds)", "avgPitstop")
display(df_race_results)





In [0]:
#Keep and cast necessary columns to integers for our model
df_race_results = df_race_results.select(["raceId","driverId","resultId","positionOrder","laps","fastestLap","fastestLapTime","fastestLapSpeed","avgPitstop","grid", "rank","fastestLapTime"])

df_race_results = df_race_results.filter((df_race_results["fastestLap"] != "\\N") & (df_race_results["fastestLapTime"] != "\\N") & (df_race_results["fastestLapSpeed"] != "\\N"))
df_race_results = df_race_results.withColumns({"raceId": col("raceId").cast("int"),
                                      "driverId": col("driverId").cast("int"),
                                      "resultId": col("resultId").cast("int"),
                                      "positionOrder": col("positionOrder").cast("int"),
                                      "laps": col("laps").cast("int"),
                                      "fastestLap": col("fastestLap").cast("int"),
                                      "fastestLapSpeed": col("fastestLapSpeed").cast("float"),
                                      "avgPitstop": col("avgPitstop").cast("float"),
                                      "grid": col("grid").cast("int"),
                                      "rank":col("rank").cast("int"),
                                     })

#Converting fastest lap time to miliseconds 
df_race_results = df_race_results.withColumn("fastestLapTime_ms",
    when(col("fastestLapTime") != "\\N",
        (regexp_extract(col("fastestLapTime"), r"(\d+):(\d+)\.(\d+)", 1).cast("int") * 60000) +  # minutes
        (regexp_extract(col("fastestLapTime"), r"(\d+):(\d+)\.(\d+)", 2).cast("int") * 1000) +  # seconds
        (regexp_extract(col("fastestLapTime"), r"(\d+):(\d+)\.(\d+)", 3).cast("int") * 1)      # milliseconds
    ).otherwise(None)
)
display(df_race_results)

In [0]:
#Creating column indicating whether driver won
df_race_results = df_race_results.withColumn("driver_won", when(col("positionOrder") == 1, 1).otherwise(0))
display(df_race_results)

In [0]:
#Converting dataset to a pandas dataset and assigning my X and y variables
df_pandas = df_race_results.toPandas()
X = df_pandas[['laps', 'fastestLap', 'fastestLapSpeed', 'avgPitstop', 'grid', 'rank', 'fastestLapTime_ms']]
y = df_pandas[['driver_won']]

#Train, test, split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

#Format y values properly
y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

##1. Creating 2 tables to store predictions later
<br>
The 2 classification models I'll be using are Random Forest, and logistic regression with penalty. 

In [0]:
#Create 2 empty tables 
#Logistic Regression table
won_predictions_logreg = X_test.copy()
won_predictions_logreg["driver_won"] = y_test
won_predictions_logreg["predictions"] = None

#Random forest table
won_predictions_rf = X_test.copy()
won_predictions_rf["driver_won"] = y_test
won_predictions_rf["predictions"] = None

display(won_predictions_rf)



##2. Setting up the ML flow for both the models and running experiments 

###Random Forest Experiment Setup

In [0]:
#-----------------Initial Random Forest Experiment-------------------------------
#Running initial flow to get experiementID to use in future random forest function

with mlflow.start_run(run_name="Initial RF Experiment") as run:
  # Create model, train it, and create predictions
  rf = RandomForestClassifier()
  rf.fit(X_train, y_train)
  #Make sure positionOrder is a whole number between 1 and 30
  predictions = rf.predict(X_test)
  
  # Log model
  mlflow.sklearn.log_model(rf, "random-forest-classifier")
  
  # Create metrics
  f1_result = f1_score(y_test, predictions)
  precision = precision_score(y_test, predictions)
  recall = recall_score(y_test, predictions)
  accuracy = accuracy_score(y_test, predictions)
  roc_auc = roc_auc_score(y_test, predictions)
  
  
  print("  f1_score: {}".format(f1_result))
  print("  precision: {}".format(precision))
  print("  recall: {}".format(recall))
  print("  accuracy: {}".format(accuracy))
  print("  roc_auc_score: {}".format(roc_auc))
  
  # Log metrics
  mlflow.log_metric("f1_score", f1_result)
  mlflow.log_metric("precision", precision)
  mlflow.log_metric("recall", recall)
  mlflow.log_metric("accuracy", accuracy)
  mlflow.log_metric("roc_auc_score", roc_auc)

  #Get runID and experimentID
  runID = run.info.run_id
  experimentID = run.info.experiment_id
  
  print("Inside MLflow Run with run_id {} and experiment_id {}".format(runID, experimentID))

In [0]:
#-----------------Define Function-------------------------------
#Defining function to run multiple experiments with ease 
def f1_rf(experimentID, run_name, params, X_train, X_test, y_train, y_test):

  with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:
    # Create model, train it, and create predictions
    rf = RandomForestClassifier(**params)
    rf.fit(X_train, y_train)
    #Make sure positionOrder is a whole number between 1 and 20
    predictions = rf.predict(X_test)

    # Log model
    mlflow.sklearn.log_model(rf, "random-forest-classifier")

    # Log params
    [mlflow.log_param(param, value) for param, value in params.items()]

    # Create metrics
    f1_result = f1_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    accuracy = accuracy_score(y_test, predictions)
    roc_auc = roc_auc_score(y_test, predictions)
    
    
    print("  f1_score: {}".format(f1_result))
    print("  precision: {}".format(precision))
    print("  recall: {}".format(recall))
    print("  accuracy: {}".format(accuracy))
    print("  roc_auc_score: {}".format(roc_auc))
    
    # Log metrics
    mlflow.log_metric("f1_score", f1_result)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("roc_auc_score", roc_auc)
    
    # Create feature importance
    importance = pd.DataFrame(list(zip(X.columns, rf.feature_importances_)), 
                                columns=["Feature", "Importance"]
                              ).sort_values("Importance", ascending=False)
    
    # Log importances using a temporary file
    temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv")
    temp_name = temp.name
    try:
      importance.to_csv(temp_name, index=False)
      mlflow.log_artifact(temp_name, "feature-importance.csv")
    finally:
      temp.close() # Delete the temp file
    
    #Create plot showing actual vs predicted values
    
    fig, ax = plt.subplots(figsize=(6, 5))

    ConfusionMatrixDisplay.from_estimator(rf, X_test, y_test, ax=ax)
    ax.set_title("Random Forest")


    #Log plot using a temporary file
    temp = tempfile.NamedTemporaryFile(prefix="confusion_matrix", suffix=".png")
    temp_name = temp.name
    try:
      fig.savefig(temp_name)
      mlflow.log_artifact(temp_name, "confusion_matrix.png")
    finally:
      temp.close() # Delete the temp file
      
    display(fig)
    return run.info.run_id

In [0]:
#-----------------Testing Outputs-------------------------------
#Testing outputs 
params = {
  "n_estimators": 100,
  "max_depth": 5,
  "random_state": 42
}

f1_rf(experimentID, "Testing Flow Outputs", params, X_train, X_test, y_train, y_test)


###Random Forest Experiments 

In [0]:
#-----------------Run 1-------------------------------
#Testing outputs 
params = {
  "n_estimators": 500,
  "max_depth": 20,
  "random_state": 42,
  "class_weight": "balanced"
}

f1_rf(experimentID, "RF_500n_20d", params, X_train, X_test, y_train, y_test)

In [0]:
#-----------------Run 2-------------------------------
#Testing outputs 
params = {
  "n_estimators": 500,
  "max_depth": 30,
  "random_state": 42,
  "class_weight": "balanced"

}

f1_rf(experimentID, "RF_500n_30d", params, X_train, X_test, y_train, y_test)

In [0]:
#-----------------Run 3-------------------------------
#Testing outputs 
params = {
  "n_estimators": 1000,
  "max_depth": 20,
  "random_state": 42,
  "class_weight": "balanced"

}

f1_rf(experimentID, "RF_1000n_20d", params, X_train, X_test, y_train, y_test)

In [0]:
#-----------------Run 4-------------------------------
#Testing outputs 
params = {
  "n_estimators": 1000,
  "max_depth": 30,
  "random_state": 42,
  "class_weight": "balanced"

}

f1_rf(experimentID, "RF_1000n_30d", params, X_train, X_test, y_train, y_test)

In [0]:
#-----------------Run 5-------------------------------
#Testing outputs 
params = {
  "n_estimators": 1500,
  "max_depth": 25,
  "random_state": 42,
 "class_weight": "balanced"

}

f1_rf(experimentID, "RF_1500n_25d", params, X_train, X_test, y_train, y_test)

###Best Model  
The highest f1 score from the 5 experiments was from run 3, with an f1-score of 0.365. This run had 1000 estimators and a max depth of 20. We will be using this model to store our predictions in the table for random forest

In [0]:
#Initialize best model
rf_best = RandomForestClassifier(n_estimators=1000, max_depth=20, random_state=42, class_weight="balanced")
#Get predictions
predictions = rf.predict(X_test)
#Store predictions into table
won_predictions_rf["predictions"] = predictions

#Convert back to pyspark database
rf_predictions_df = spark.createDataFrame(won_predictions_rf)

  

In [0]:
#Write predictions table to csv
rf_predictions_df.write.csv("/Volumes/gr5069/sbr2174/assignments/homework_4/random_forest_pred.csv")

###Logistic Regression Experiment Setup

In [0]:
#Using standard scaler to standardize dependent variables 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [0]:
#-----------------Initial Logistic Experiment-------------------------------
#Running initial flow to get experiementID to use in future log function

with mlflow.start_run(run_name="Initial Log Experiment") as run:
  # Create model, train it, and create predictions
  lg = LogisticRegression()
  lg.fit(X_train_scaled, y_train)
  #Make sure positionOrder is a whole number between 1 and 30
  predictions = lg.predict(X_test_scaled)
  
  # Log model
  mlflow.sklearn.log_model(lg, "logistic-regression")
  
  # Create metrics
  f1_result = f1_score(y_test, predictions)
  precision = precision_score(y_test, predictions)
  recall = recall_score(y_test, predictions)
  accuracy = accuracy_score(y_test, predictions)
  roc_auc = roc_auc_score(y_test, predictions)
  
  
  print("  f1_score: {}".format(f1_result))
  print("  precision: {}".format(precision))
  print("  recall: {}".format(recall))
  print("  accuracy: {}".format(accuracy))
  print("  roc_auc_score: {}".format(roc_auc))
  
  # Log metrics
  mlflow.log_metric("f1_score", f1_result)
  mlflow.log_metric("precision", precision)
  mlflow.log_metric("recall", recall)
  mlflow.log_metric("accuracy", accuracy)
  mlflow.log_metric("roc_auc_score", roc_auc)

  #Get runID and experimentID
  runID = run.info.run_id
  experimentID = run.info.experiment_id
  
  print("Inside MLflow Run with run_id {} and experiment_id {}".format(runID, experimentID))

In [0]:
#-----------------Define Function-------------------------------
#Defining function to run multiple experiments with ease 
def f1_lg(experimentID, run_name, params, X_train, X_test, y_train, y_test):

  with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:
    # Create model, train it, and create predictions
    lg = LogisticRegression(**params)
    lg.fit(X_train, y_train)
    #Make sure positionOrder is a whole number between 1 and 20
    predictions = lg.predict(X_test)

    # Log model
    mlflow.sklearn.log_model(lg, "logistic-regression")

    # Log params
    [mlflow.log_param(param, value) for param, value in params.items()]

    # Create metrics
    f1_result = f1_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    accuracy = accuracy_score(y_test, predictions)
    roc_auc = roc_auc_score(y_test, predictions)
    
    
    print("  f1_score: {}".format(f1_result))
    print("  precision: {}".format(precision))
    print("  recall: {}".format(recall))
    print("  accuracy: {}".format(accuracy))
    print("  roc_auc_score: {}".format(roc_auc))
    
    # Log metrics
    mlflow.log_metric("f1_score", f1_result)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("roc_auc_score", roc_auc)
    
    # Create feature importance
    # importance = pd.DataFrame(list(zip(X.columns, lg.coef_)), 
    #                             columns=["Feature", "Importance"]
    #                           ).sort_values("Importance", ascending=False)

    importance = pd.DataFrame({"Feature": X.columns, "Importance": abs(lg.coef_[0])}).sort_values("Importance")
    
    # Log importances using a temporary file
    temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv")
    temp_name = temp.name
    try:
      importance.to_csv(temp_name, index=False)
      mlflow.log_artifact(temp_name, "feature-importance.csv")
    finally:
      temp.close() # Delete the temp file
    
    #Create plot showing actual vs predicted values
    
    fig, ax = plt.subplots(figsize=(6, 5))

    ConfusionMatrixDisplay.from_estimator(lg, X_test, y_test, ax=ax)
    ax.set_title("Logistic Regression")


    #Log plot using a temporary file
    temp = tempfile.NamedTemporaryFile(prefix="confusion_matrix", suffix=".png")
    temp_name = temp.name
    try:
      fig.savefig(temp_name)
      mlflow.log_artifact(temp_name, "confusion_matrix.png")
    finally:
      temp.close() # Delete the temp file
      
    display(fig)
    return run.info.run_id

In [0]:
#-----------------Testing Outputs-------------------------------
#Testing outputs 
params = {
  "C": 1,
  "penalty": 'l2',
  "max_iter": 1000,
  "class_weight": "balanced"
}

f1_lg(experimentID, "Testing Flow Outputs", params, X_train_scaled, X_test_scaled, y_train, y_test)

###Logistic Regression Experiments

In [0]:
#-----------------Run 1-------------------------------
#Testing outputs 
params = {
  "C": 1,
  "penalty": 'l2',
  "max_iter": 1000,
  "class_weight": "balanced"
}

f1_lg(experimentID, "lg_1c", params, X_train_scaled, X_test_scaled, y_train, y_test)

In [0]:
#-----------------Run 2-------------------------------
#Testing outputs 
params = {
  "C": 10,
  "penalty": 'l2',
  "max_iter": 1000,
  "class_weight": "balanced"
}

f1_lg(experimentID, "lg_10c", params, X_train_scaled, X_test_scaled, y_train, y_test)

In [0]:
#-----------------Run 3-------------------------------
#Testing outputs 
params = {
  "C": 100,
  "penalty": 'l2',
  "max_iter": 1000,
  "class_weight": "balanced"
}

f1_lg(experimentID, "lg_100c", params, X_train_scaled, X_test_scaled, y_train, y_test)

In [0]:
#-----------------Run 4-------------------------------
#Testing outputs 
params = {
  "C": .1,
  "penalty": 'l2',
  "max_iter": 1000,
  "class_weight": "balanced"
}

f1_lg(experimentID, "lg_.1c", params, X_train_scaled, X_test_scaled, y_train, y_test)

In [0]:
#-----------------Run 5-------------------------------
#Testing outputs 
params = {
  "C": .001,
  "penalty": 'l2',
  "max_iter": 1000,
  "class_weight": "balanced"
}

f1_lg(experimentID, "lg_.001c", params, X_train_scaled, X_test_scaled, y_train, y_test)

###Best Model  
The best model for our logistic experiment runs was run 2, with an f1-score of 0.405. The main parameter was a C of 10, this is a constraint on our coefficients. Run 3 had a similar result, but this may be due to the lower impact C has on coefficients as it increases 

In [0]:
#Initializing best model
lg_best = LogisticRegression(C=10, penalty='l2', max_iter=1000, class_weight="balanced")
#Get predictions
predictions = lg.predict(X_test_scaled)
#Store predictions in table
won_predictions_logreg["predictions"] = predictions

#Convert table back to pyspark
lg_predictions_df = spark.createDataFrame(won_predictions_rf)

In [0]:
#Write predictions to csv
lg_predictions_df.write.csv("/Volumes/gr5069/sbr2174/assignments/homework_4/logistic_pred.csv")